# Notebook 1 — Fetch & Validate Nifty Sector Data

This notebook downloads historical daily price data for 10 Nifty sector indices using `yfinance`.

**What it does:**
- Downloads up to 10 years of data per sector (uses whatever is available if less)
- Cleans and validates each dataset
- Saves each sector as a CSV in `data/raw/`
- Produces a summary report of what data we have

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# Uncomment if you haven't installed these yet
# !pip install yfinance pandas openpyxl

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [8]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Date range — try to go back 10 years, use whatever is available
END_DATE   = datetime.today().strftime('%Y-%m-%d')
START_DATE = (datetime.today() - timedelta(days=365 * 10)).strftime('%Y-%m-%d')

print(f'Requesting data from {START_DATE} to {END_DATE}')

# Sector index tickers on yfinance
# Format: { 'Sector Name': 'yfinance_ticker' }
SECTORS = {
    'Nifty IT'                 : '^CNXIT',
    'Nifty Bank'               : '^NSEBANK',
    'Nifty Auto'               : '^CNXAUTO',
    'Nifty FMCG'               : '^CNXFMCG',
    'Nifty Pharma'             : '^CNXPHARMA',
    'Nifty Metal'              : '^CNXMETAL',
    'Nifty Realty'             : '^CNXREALTY',
    'Nifty Energy'             : '^CNXENERGY',
    'Nifty Financial Services' : 'NIFTY_FIN_SERVICE.NS',
    'Nifty Media'              : '^CNXMEDIA',
}

# Output folder
RAW_DATA_DIR = 'data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)
print(f'Data will be saved to: {RAW_DATA_DIR}/')

Requesting data from 2016-05-26 to 2026-05-24
Data will be saved to: data/raw/


In [9]:
# ── Download & Save ───────────────────────────────────────────────────────────
# We download each sector one by one so we can handle failures gracefully.
# If a ticker fails or returns empty data, we log it and move on.

summary = []  # Will collect a report row per sector

for sector_name, ticker in SECTORS.items():
    print(f'\nFetching: {sector_name} ({ticker})')
    
    try:
        # Download OHLCV data
        raw = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            progress=False,   # suppress yfinance progress bar
            auto_adjust=True  # adjusts for splits/dividends automatically
        )
        
        # ── Validate ──────────────────────────────────────────────────────────
        if raw.empty:
            print(f'  ⚠ No data returned for {ticker}. Skipping.')
            summary.append({
                'Sector'      : sector_name,
                'Ticker'      : ticker,
                'Status'      : 'FAILED — no data',
                'Start Date'  : None,
                'End Date'    : None,
                'Total Days'  : 0,
                'Years Approx': 0,
                'Missing %'   : None,
            })
            continue
        
        # Flatten multi-level columns if present (yfinance sometimes returns them)
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        
        # Keep only relevant columns
        df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        df.index.name = 'Date'
        
        # ── Check for missing values ───────────────────────────────────────────
        missing_count = df['Close'].isna().sum()
        missing_pct   = round(missing_count / len(df) * 100, 2)
        
        # Forward-fill small gaps (holidays, etc.) — max 3 consecutive days
        df = df.ffill(limit=3)
        
        # Drop any remaining NaNs
        df = df.dropna()
        
        # ── Save to CSV ────────────────────────────────────────────────────────
        # Filename uses sector name with underscores
        filename   = sector_name.replace(' ', '_').lower() + '.csv'
        filepath   = os.path.join(RAW_DATA_DIR, filename)
        df.to_csv(filepath)
        
        actual_start = df.index.min().strftime('%Y-%m-%d')
        actual_end   = df.index.max().strftime('%Y-%m-%d')
        years_approx = round(len(df) / 252, 1)  # ~252 trading days per year
        
        print(f'  ✓ {len(df)} trading days | {actual_start} → {actual_end} | ~{years_approx} yrs | Missing: {missing_pct}%')
        
        summary.append({
            'Sector'      : sector_name,
            'Ticker'      : ticker,
            'Status'      : 'OK',
            'Start Date'  : actual_start,
            'End Date'    : actual_end,
            'Total Days'  : len(df),
            'Years Approx': years_approx,
            'Missing %'   : missing_pct,
        })
    
    except Exception as e:
        print(f'  ✗ Error fetching {ticker}: {e}')
        summary.append({
            'Sector'      : sector_name,
            'Ticker'      : ticker,
            'Status'      : f'ERROR — {str(e)}',
            'Start Date'  : None,
            'End Date'    : None,
            'Total Days'  : 0,
            'Years Approx': 0,
            'Missing %'   : None,
        })

print('\n── Download complete ──')


Fetching: Nifty IT (^CNXIT)
  ✓ 2464 trading days | 2016-05-26 → 2026-05-22 | ~9.8 yrs | Missing: 0.0%

Fetching: Nifty Bank (^NSEBANK)
  ✓ 2464 trading days | 2016-05-26 → 2026-05-22 | ~9.8 yrs | Missing: 0.0%

Fetching: Nifty Auto (^CNXAUTO)
  ✓ 2449 trading days | 2016-05-26 → 2026-05-22 | ~9.7 yrs | Missing: 0.0%

Fetching: Nifty FMCG (^CNXFMCG)
  ✓ 2448 trading days | 2016-05-26 → 2026-05-22 | ~9.7 yrs | Missing: 0.0%

Fetching: Nifty Pharma (^CNXPHARMA)
  ✓ 2463 trading days | 2016-05-26 → 2026-05-22 | ~9.8 yrs | Missing: 0.0%

Fetching: Nifty Metal (^CNXMETAL)
  ✓ 2449 trading days | 2016-05-26 → 2026-05-22 | ~9.7 yrs | Missing: 0.0%

Fetching: Nifty Realty (^CNXREALTY)
  ✓ 2449 trading days | 2016-05-26 → 2026-05-22 | ~9.7 yrs | Missing: 0.0%

Fetching: Nifty Energy (^CNXENERGY)
  ✓ 2449 trading days | 2016-05-26 → 2026-05-22 | ~9.7 yrs | Missing: 0.0%

Fetching: Nifty Financial Services (NIFTY_FIN_SERVICE.NS)
  ✓ 2449 trading days | 2016-05-26 → 2026-05-22 | ~9.7 yrs | Missin

In [10]:
# ── Summary Table ─────────────────────────────────────────────────────────────
# Shows exactly what data we have per sector before moving to the scanner.

summary_df = pd.DataFrame(summary)

print('\n════════════════════════════════════════════════════════')
print('           SECTOR DATA AVAILABILITY SUMMARY')
print('════════════════════════════════════════════════════════')
print(summary_df.to_string(index=False))

# Separate OK vs failed
ok_sectors     = summary_df[summary_df['Status'] == 'OK']
failed_sectors = summary_df[summary_df['Status'] != 'OK']

print(f'\n✓ Successfully downloaded: {len(ok_sectors)} sectors')
if len(failed_sectors) > 0:
    print(f'✗ Failed: {len(failed_sectors)} sectors → {list(failed_sectors["Sector"])}')
    print('  These will be skipped in the scanner. You can try alternate tickers for them.')


════════════════════════════════════════════════════════
           SECTOR DATA AVAILABILITY SUMMARY
════════════════════════════════════════════════════════
                  Sector               Ticker Status Start Date   End Date  Total Days  Years Approx  Missing %
                Nifty IT               ^CNXIT     OK 2016-05-26 2026-05-22        2464           9.8        0.0
              Nifty Bank             ^NSEBANK     OK 2016-05-26 2026-05-22        2464           9.8        0.0
              Nifty Auto             ^CNXAUTO     OK 2016-05-26 2026-05-22        2449           9.7        0.0
              Nifty FMCG             ^CNXFMCG     OK 2016-05-26 2026-05-22        2448           9.7        0.0
            Nifty Pharma           ^CNXPHARMA     OK 2016-05-26 2026-05-22        2463           9.8        0.0
             Nifty Metal            ^CNXMETAL     OK 2016-05-26 2026-05-22        2449           9.7        0.0
            Nifty Realty           ^CNXREALTY     OK 2016

In [11]:
# ── Quick Visual Check ────────────────────────────────────────────────────────
# Load one sector back and print the first & last 5 rows to confirm data looks clean.

CHECK_SECTOR = 'Nifty IT'  # Change this to inspect any sector

check_file = os.path.join(RAW_DATA_DIR, CHECK_SECTOR.replace(' ', '_').lower() + '.csv')

if os.path.exists(check_file):
    check_df = pd.read_csv(check_file, index_col='Date', parse_dates=True)
    print(f'--- {CHECK_SECTOR} — First 5 rows ---')
    print(check_df.head())
    print(f'\n--- {CHECK_SECTOR} — Last 5 rows ---')
    print(check_df.tail())
    print(f'\nShape: {check_df.shape}')
    print(f'Any NaN: {check_df.isna().any().any()}')
else:
    print(f'File not found for {CHECK_SECTOR} — it may have failed to download.')

--- Nifty IT — First 5 rows ---
                    Open          High           Low         Close  Volume
Date                                                                      
2016-05-26  11231.849609  11366.049805  11178.250000  11337.150391   15800
2016-05-27  11325.700195  11462.049805  11325.700195  11420.849609   11300
2016-05-30  11433.950195  11579.250000  11433.950195  11547.500000   10200
2016-05-31  11554.650391  11559.450195  11374.849609  11395.849609   32100
2016-06-01  11405.500000  11516.000000  11405.500000  11494.900391   10800

--- Nifty IT — Last 5 rows ---
                    Open          High           Low         Close  Volume
Date                                                                      
2026-05-18  27689.599609  28466.099609  27663.800781  28389.800781   47300
2026-05-19  28679.849609  29609.599609  28677.800781  29308.000000   86700
2026-05-20  29140.800781  29517.150391  29036.650391  29185.150391   44300
2026-05-21  29277.949219  29328.3496

In [12]:
# ── Save Summary ──────────────────────────────────────────────────────────────
# Save the summary table so Notebook 2 knows which sectors are available
# and how many years of data each has.

summary_path = os.path.join('data', 'sector_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f'Summary saved to: {summary_path}')
print('\n✅ Notebook 1 complete. Proceed to 02_scanner.ipynb')

Summary saved to: data\sector_summary.csv

✅ Notebook 1 complete. Proceed to 02_scanner.ipynb
